# Wave Modeling of the Solar Wind — Implementation / 구현

**Paper**: Ofman, L. (2010). *Wave Modeling of the Solar Wind*. Living Reviews in Solar Physics, 7, 4. DOI: 10.12942/lrsp-2010-4

이 노트북은 Ofman (2010) 리뷰의 **핵심 파동 구동 태양풍 개념**들을 선택적으로 구현합니다. 리뷰이므로 전체를 재현할 수 없고, 각 접근(Parker 고전, WKB 파동 구동, 이온-사이클로트론, 반사 난류)의 **최소 작동 버전**을 NumPy로 구현하여 관측과 비교합니다.

This notebook selectively implements the key concepts from Ofman (2010). For each family — Parker's classical wind, WKB wave-driven wind, ion-cyclotron resonance, reflection-driven turbulence — we build a **minimal working version** in NumPy and compare to observations.

**Contents / 구성:**
1. Parker's isothermal wind — classical solution / Parker 고전 해
2. WKB wave amplitude growth along the flow / WKB 파동 진폭 성장
3. Wave-driven wind: integrated momentum equation / 파동 구동 바람
4. Ion-cyclotron resonance frequencies / 이온-사이클로트론 공명
5. Elsässer variables & turbulent cascade / Elsässer 변수와 난류
6. Reflection-driven turbulence heating profile / 반사 난류 가열
7. Spectral scaling: Kolmogorov vs. Iroshnikov-Kraichnan / 스펙트럼 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

R_SUN = 6.96e8
M_SUN = 1.989e30
G_NEWT = 6.674e-11
K_B = 1.381e-23
M_P = 1.673e-27
MU0 = 4 * np.pi * 1e-7
E_CHARGE = 1.602e-19
GM = G_NEWT * M_SUN

## Part 1: Parker's Isothermal Wind / Parker 등온 태양풍

The classical isothermal wind (§3 of Ofman 2010):

$$
\frac{1}{u}\frac{du}{dr}\bigl(u^2 - c_s^2\bigr) = \frac{2c_s^2}{r} - \frac{GM_\odot}{r^2}
$$

Physical solution passes the sonic critical point at $r_c = GM_\odot / (2c_s^2)$. We solve by integrating outward from $r_c$.

등온 가정의 고전 바람. 임계점 $r_c$를 통과하는 해가 관측되는 태양풍. $T=10^6$ K에서 1 AU 속도는 약 ~400 km/s로 포화 — 빠른 바람 설명 불가.

In [ ]:
def parker_wind_speed(r: np.ndarray, T: float) -> np.ndarray:
    """Return Parker isothermal wind speed along r using the implicit algebraic form.

    Solves (u/c_s)^2 - ln(u/c_s)^2 = 4 ln(r/r_c) + 4 r_c/r - 3 on the outer branch.

    Args:
        r: Radii in meters, shape (N,). Must satisfy r > 0.
        T: Isothermal temperature (K).

    Returns:
        Wind speed array in m/s, same shape as r.
    """
    from scipy.optimize import brentq

    c_s = np.sqrt(2 * K_B * T / M_P)
    r_c = GM / (2 * c_s**2)

    def f(v, x):
        return v**2 - 2 * np.log(v) - 4 * np.log(x) - 4 / x + 3

    u = np.zeros_like(r)
    for i, ri in enumerate(r):
        x = ri / r_c
        if x < 1:
            u[i] = c_s * brentq(lambda v: f(v, x), 1e-4, 0.9999)
        elif x == 1:
            u[i] = c_s
        else:
            u[i] = c_s * brentq(lambda v: f(v, x), 1.0001, 20)
    return u


r_arr = np.linspace(1.0, 215.0, 100) * R_SUN

fig, ax = plt.subplots(figsize=(10, 6))
for T in [0.5e6, 1.0e6, 1.5e6, 2.0e6]:
    u = parker_wind_speed(r_arr, T) / 1e3
    ax.plot(r_arr / R_SUN, u, label=f'T = {T/1e6:.1f} MK')
ax.axhline(400, color='gray', ls=':', label='slow wind (~400 km/s)')
ax.axhline(750, color='red', ls=':', label='fast wind (~750 km/s)')
ax.set_xlabel('r / R_sun')
ax.set_ylabel('u (km/s)')
ax.set_title('Parker isothermal wind — fast wind is unreachable')
ax.legend()
ax.set_xscale('log')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Part 2: WKB Wave-Amplitude Growth along the Flow / WKB 파동 진폭 성장

Wave-action conservation (§4):

$$
\frac{d}{dr}\!\left[\langle\delta B^2\rangle\,\frac{(u+v_A)^2}{v_A}\,A(r)\right] = 0
$$

For a super-radial flux tube $A(r) = A_0(r/R_\odot)^2 f(r)$ with expansion $f(r) > 1$. Given a background density and $B(r)$ profile, we can compute $\delta B(r)$ analytically.

파동 작용이 보존되면, 밀도가 떨어지고 $v_A$가 변하는 흐름을 따라 파동 진폭이 **증폭**. 이것이 바람 가속의 연료.

In [ ]:
def wkb_wave_amplitude(
    r: np.ndarray,
    u: np.ndarray,
    rho: np.ndarray,
    B: np.ndarray,
    A: np.ndarray,
    dB2_0: float,
) -> np.ndarray:
    """Compute <δB^2>(r) from wave-action conservation along a flux tube.

    Args:
        r: Radii (m).
        u: Flow speed (m/s).
        rho: Mass density (kg/m^3).
        B: Background magnetic field (T).
        A: Flux tube cross-section (m^2).
        dB2_0: Wave energy <δB^2> at the first grid point.

    Returns:
        Wave energy array <δB^2>(r).
    """
    v_A = B / np.sqrt(MU0 * rho)
    F0 = dB2_0 * (u[0] + v_A[0]) ** 2 / v_A[0] * A[0]
    return F0 * v_A / ((u + v_A) ** 2 * A)


r = np.linspace(1.01, 215.0, 200) * R_SUN
u = np.linspace(50e3, 700e3, 200)
rho = 1e-12 * (R_SUN / r) ** 5
B = 1e-4 * (R_SUN / r) ** 2
A = 4 * np.pi * r**2 * (1 + 5 * np.exp(-((r / R_SUN - 1) / 0.5) ** 2))

dB2 = wkb_wave_amplitude(r, u, rho, B, A, dB2_0=(0.1 * B[0]) ** 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].loglog(r / R_SUN, np.sqrt(dB2) / B, label=r'$\delta B / B_0$')
axes[0].set_xlabel('r / R_sun')
axes[0].set_ylabel(r'relative wave amplitude')
axes[0].set_title('WKB Alfvén wave amplitude grows with distance')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

v_A = B / np.sqrt(MU0 * rho)
axes[1].loglog(r / R_SUN, v_A / 1e3, label='v_A')
axes[1].loglog(r / R_SUN, u / 1e3, label='u')
idx_alfven = np.argmin(np.abs(u - v_A))
axes[1].axvline(r[idx_alfven] / R_SUN, color='red', ls=':', label=f'Alfvén surface ≈ {r[idx_alfven]/R_SUN:.1f} R_sun')
axes[1].set_xlabel('r / R_sun')
axes[1].set_ylabel('speed (km/s)')
axes[1].set_title('Alfvén speed vs. flow speed')
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Part 3: Wave-Driven Wind — Integrated Momentum Equation / 파동 구동 바람

Adding the wave-pressure force to Parker:

$$
\rho u\frac{du}{dr} = -\frac{dp}{dr} - \frac{\rho GM_\odot}{r^2} - \frac{1}{2}\frac{d}{dr}\!\left(\frac{\langle\delta B^2\rangle}{\mu_0}\right)
$$

We integrate a simplified version and compare to Parker to show that wave pressure **adds ~hundreds of km/s** of asymptotic speed.

Parker 해에 파동 압력 항을 추가하면 1 AU 속도가 **수백 km/s 증가**. 빠른 바람의 원천 — §4의 핵심 메시지.

In [ ]:
def wave_driven_wind_simple(
    r_end: float = 215.0,
    T: float = 1e6,
    dB0_over_B: float = 0.1,
    rho0: float = 1e-12,
    B0: float = 1e-4,
) -> tuple:
    """Integrate a simplified wave-driven wind momentum equation.

    Assumes isothermal gas + WKB Alfvén wave pressure on a 1/r^2 flux tube.

    Args:
        r_end: Outer boundary in units of R_sun.
        T: Isothermal temperature (K).
        dB0_over_B: Relative wave amplitude at r=R_sun.
        rho0: Base density at R_sun (kg/m^3).
        B0: Base magnetic field (T).

    Returns:
        Tuple (r/R_sun array, u(r) in m/s).
    """
    c_s = np.sqrt(2 * K_B * T / M_P)
    r_c = GM / (2 * c_s**2) / R_SUN

    def rhs(r_rs, u):
        r = r_rs * R_SUN
        rho = rho0 * (R_SUN / r) ** 5
        B = B0 * (R_SUN / r) ** 2
        v_A = B / np.sqrt(MU0 * rho)
        # δB^2 from WKB with A ∝ r^2
        dB2_0 = (dB0_over_B * B0) ** 2
        F0 = dB2_0 * (u[0] + np.sqrt(B0**2 / (MU0 * rho0))) ** 2 / np.sqrt(B0**2 / (MU0 * rho0)) * R_SUN**2
        dB2 = F0 * v_A / ((u + v_A) ** 2 * r**2)
        # d(δB^2)/dr via finite difference-like analytic derivative
        eps = 1e-4
        r_plus = r * (1 + eps)
        rho_p = rho0 * (R_SUN / r_plus) ** 5
        B_p = B0 * (R_SUN / r_plus) ** 2
        v_A_p = B_p / np.sqrt(MU0 * rho_p)
        dB2_p = F0 * v_A_p / ((u + v_A_p) ** 2 * r_plus**2)
        dpw_dr = (dB2_p - dB2) / (r_plus - r) / MU0

        num = 2 * c_s**2 / r_rs - GM / (R_SUN * r_rs**2) / R_SUN - 0.5 * dpw_dr / rho
        den = u[0] - c_s**2 / u[0]
        return [num / den]

    sol = solve_ivp(
        rhs, (r_c + 0.01, r_end), [c_s * 1.001], max_step=0.5, rtol=1e-6, atol=1e-3,
    )
    return sol.t, sol.y[0]


r_wkb, u_wkb = wave_driven_wind_simple(T=1e6, dB0_over_B=0.1)
r_parker = np.linspace(1.0, 215.0, 100) * R_SUN
u_parker = parker_wind_speed(r_parker, 1e6)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r_parker / R_SUN, u_parker / 1e3, label='Parker (no waves)', lw=2)
ax.plot(r_wkb, u_wkb / 1e3, label='Wave-driven (δB/B=0.1)', lw=2)
for dB_rel, color in zip([0.05, 0.15, 0.2], ['C2', 'C3', 'C4']):
    r_w, u_w = wave_driven_wind_simple(T=1e6, dB0_over_B=dB_rel)
    ax.plot(r_w, u_w / 1e3, '--', label=f'δB/B = {dB_rel}', color=color, alpha=0.7)
ax.axhline(400, color='gray', ls=':', alpha=0.6)
ax.axhline(750, color='red', ls=':', alpha=0.6)
ax.set_xlabel('r / R_sun')
ax.set_ylabel('u (km/s)')
ax.set_title('Wave pressure boosts asymptotic speed — fast wind reachable')
ax.set_xscale('log')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Part 4: Ion-Cyclotron Resonance Frequencies / 이온-사이클로트론 공명 주파수

Resonance condition (§6):

$$
\omega - k_\parallel v_\parallel = n\,\Omega_i, \quad \Omega_i = \frac{q_iB}{m_i}
$$

We compute the proton, alpha, O⁵⁺ gyrofrequencies along the corona. The **frequency gap** between photospheric injection (~10⁻³ Hz) and ion scales (~10² Hz at 1 $R_\odot$) shows why a turbulent cascade is required.

광구 주입(~10⁻³ Hz)과 이온 자이로(~10² Hz) 사이 6 decade 갭 — 난류 캐스케이드의 필요성을 시각화.

In [ ]:
def gyrofrequency(B: np.ndarray, q_over_m: float) -> np.ndarray:
    """Return ion gyrofrequency Ω_i/(2π) in Hz.

    Args:
        B: Magnetic field in Tesla.
        q_over_m: Charge-to-mass ratio in C/kg.

    Returns:
        Gyrofrequency in Hz.
    """
    return np.abs(q_over_m * B) / (2 * np.pi)


r = np.linspace(1.0, 50.0, 300)
B = 1e-4 * (1 / r) ** 2

species = {
    'H$^+$ (proton)': E_CHARGE / M_P,
    r'He$^{2+}$ (alpha)': 2 * E_CHARGE / (4 * M_P),
    r'O$^{5+}$': 5 * E_CHARGE / (16 * M_P),
    'Fe$^{10+}$': 10 * E_CHARGE / (56 * M_P),
}

fig, ax = plt.subplots(figsize=(10, 6))
for label, qm in species.items():
    ax.loglog(r, gyrofrequency(B, qm), label=label)
ax.axhline(1e-3, color='k', ls='--', label='photospheric injection (~10$^{-3}$ Hz)')
ax.fill_between(r, 1e-3, gyrofrequency(B, E_CHARGE / M_P), alpha=0.1, color='orange',
                label='cascade range')
ax.set_xlabel('r / R_sun')
ax.set_ylabel('gyrofrequency (Hz)')
ax.set_title('Frequency gap: photospheric injection vs. ion cyclotron scales')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Part 5: Elsässer Variables from a Synthetic Alfvén Spectrum / 합성 Alfvén 스펙트럼으로부터 Elsässer 변수

We generate a synthetic outward-dominated Alfvén spectrum ($|z^+|\gg|z^-|$) and visualize the velocity and magnetic-field components. This mirrors the Belcher-Davis (1971) observation that the solar wind is Alfvénic.

외향 지배 Alfvén 스펙트럼을 합성하여 Belcher-Davis 관측을 흉내. $\delta u$와 $\delta B/\sqrt{\mu_0\rho}$가 **1:1 상관관계**를 가짐을 확인.

In [ ]:
def synthesize_alfvenic_fluctuations(
    t: np.ndarray,
    v_A: float,
    outward_fraction: float = 0.9,
    spectral_slope: float = -5 / 3,
    n_modes: int = 200,
    seed: int = 1,
) -> tuple:
    """Generate outward-dominated Alfvénic fluctuations with a chosen power spectrum.

    Args:
        t: Time array (s).
        v_A: Alfvén speed (m/s).
        outward_fraction: Ratio of z^+ to total.
        spectral_slope: Power spectral slope (e.g., -5/3 Kolmogorov).
        n_modes: Number of Fourier modes.
        seed: Random seed.

    Returns:
        Tuple (u_perp, b_over_sqrt_mu_rho) time series — Alfvén units.
    """
    rng = np.random.default_rng(seed)
    f = np.linspace(0.01, 1.0, n_modes)
    amps = f ** (spectral_slope / 2)
    phi_p = rng.uniform(0, 2 * np.pi, n_modes)
    phi_m = rng.uniform(0, 2 * np.pi, n_modes)
    z_plus = np.zeros_like(t)
    z_minus = np.zeros_like(t)
    for i in range(n_modes):
        z_plus += amps[i] * np.sin(2 * np.pi * f[i] * t + phi_p[i])
        z_minus += amps[i] * np.sin(2 * np.pi * f[i] * t + phi_m[i])
    scale = 1.0
    z_plus *= outward_fraction * scale
    z_minus *= (1 - outward_fraction) * scale
    u = 0.5 * (z_plus + z_minus)
    b = 0.5 * (z_plus - z_minus)
    return u, b


t = np.linspace(0, 500, 4000)
u_p, b_p = synthesize_alfvenic_fluctuations(t, v_A=5e5, outward_fraction=0.9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(t[:500], u_p[:500], label='δu')
axes[0].plot(t[:500], b_p[:500], label=r'δb/$\sqrt{\mu_0\rho}$', alpha=0.8)
axes[0].set_xlabel('time (s)')
axes[0].set_ylabel('fluctuation (arb.)')
axes[0].set_title('Outward-dominated Alfvénic time series')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].scatter(u_p, b_p, s=2, alpha=0.5)
axes[1].set_xlabel('δu')
axes[1].set_ylabel(r'δb/$\sqrt{\mu_0\rho}$')
axes[1].set_title('Belcher-Davis correlation: 1:1 slope for Alfvénic wind')
axes[1].grid(alpha=0.3)
axes[1].set_aspect('equal', 'box')
plt.tight_layout()
plt.show()

corr = np.corrcoef(u_p, b_p)[0, 1]
print(f'Pearson correlation (δu, δb) = {corr:.3f} — Alfvénic if ≈1')

## Part 6: Reflection-Driven Turbulence Heating Profile / 반사 난류 가열

Following Cranmer & van Ballegooijen (2005), the heating rate per unit mass (§7):

$$
Q \propto \rho\,\langle z^+ z^-\rangle\,\frac{|z^+|+|z^-|}{2\lambda_\perp}
$$

The reflection coefficient $R \sim |\partial_r v_A|/|\omega|$ sets $|z^-|/|z^+|$. We plot $Q(r)$ along a flux tube and compare to the SOHO/UVCS-inspired empirical heating profile.

$z^-$가 $z^+$로부터 반사되어 두 성분이 **동시 존재**할 때만 가열이 발생. $v_A$가 급변하는 근태양 영역이 주요 가열 영역.

In [ ]:
def reflection_driven_heating(r: np.ndarray, rho: np.ndarray, B: np.ndarray,
                              omega: float, lambda_perp: float, z_plus_amp: float) -> np.ndarray:
    """Compute reflection-driven turbulent heating rate Q(r) per unit volume.

    Args:
        r: Radii (m).
        rho: Density profile (kg/m^3).
        B: Magnetic field profile (T).
        omega: Characteristic Alfvén wave frequency (1/s).
        lambda_perp: Transverse correlation length (m).
        z_plus_amp: Outward Elsässer amplitude (m/s).

    Returns:
        Heating rate Q(r) in W/m^3.
    """
    v_A = B / np.sqrt(MU0 * rho)
    dvA_dr = np.gradient(v_A, r)
    R_coef = np.abs(dvA_dr) / max(omega, 1e-6)
    R_coef = np.clip(R_coef, 0, 1)
    z_minus = R_coef * z_plus_amp
    Q = rho * z_plus_amp * z_minus * (z_plus_amp + z_minus) / (2 * lambda_perp)
    return Q


r = np.linspace(1.01, 50.0, 500) * R_SUN
rho = 1e-12 * (R_SUN / r) ** 5
B = 1e-4 * (R_SUN / r) ** 2
Q = reflection_driven_heating(r, rho, B, omega=2 * np.pi * 1e-3,
                              lambda_perp=1e7, z_plus_amp=5e4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].loglog(r / R_SUN, Q, label='Q(r)')
axes[0].set_xlabel('r / R_sun')
axes[0].set_ylabel('volumetric heating rate (W/m³)')
axes[0].set_title('Reflection-driven turbulent heating')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

v_A = B / np.sqrt(MU0 * rho)
dvA_dr = np.gradient(v_A, r)
axes[1].loglog(r / R_SUN, np.abs(dvA_dr), label=r'$|\partial_r v_A|$')
axes[1].set_xlabel('r / R_sun')
axes[1].set_ylabel(r'$|\partial_r v_A|$ (1/s)')
axes[1].set_title('Reflection driver: Alfvén speed gradient')
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Part 7: Spectral Scaling — Kolmogorov vs. Iroshnikov-Kraichnan / 스펙트럼 비교

Build two synthetic cascades and measure their spectra:
- Kolmogorov: $E(k)\propto k^{-5/3}$
- Iroshnikov-Kraichnan: $E(k)\propto k^{-3/2}$

관측된 태양풍 스펙트럼(~$f^{-5/3}$)은 Kolmogorov에 가까우나 영역·스케일 의존성이 있음.

In [ ]:
def synthesize_power_law_series(N: int, dt: float, slope: float, seed: int = 0) -> np.ndarray:
    """Return a real time series whose PSD has the requested slope.

    Args:
        N: Number of samples.
        dt: Sampling step (s).
        slope: Target slope of the energy spectrum (e.g., -5/3).
        seed: Random seed.

    Returns:
        Time series of length N.
    """
    rng = np.random.default_rng(seed)
    f = np.fft.rfftfreq(N, dt)
    amp = np.zeros_like(f)
    amp[1:] = f[1:] ** (slope / 2)
    phase = np.exp(1j * rng.uniform(0, 2 * np.pi, len(f)))
    spec = amp * phase
    return np.fft.irfft(spec, n=N)


N = 2**14
dt = 1.0
x_kol = synthesize_power_law_series(N, dt, slope=-5 / 3, seed=1)
x_ik = synthesize_power_law_series(N, dt, slope=-3 / 2, seed=2)

def psd(x, dt):
    fft = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), dt)
    p = np.abs(fft) ** 2 / len(x)
    return f[1:], p[1:]


f1, p1 = psd(x_kol, dt)
f2, p2 = psd(x_ik, dt)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(f1, p1, alpha=0.7, label='synthetic Kolmogorov')
ax.loglog(f2, p2, alpha=0.7, label='synthetic Iroshnikov-Kraichnan')
ax.loglog(f1, f1**(-5/3) * p1[10] / f1[10]**(-5/3), 'k--', label=r'$f^{-5/3}$ reference')
ax.loglog(f2, f2**(-3/2) * p2[10] / f2[10]**(-3/2), 'k:', label=r'$f^{-3/2}$ reference')
ax.set_xlabel('f (Hz)')
ax.set_ylabel('PSD')
ax.set_title('Kolmogorov vs. Iroshnikov-Kraichnan spectra')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | In Ofman 2010 | Our implementation | Modern tools / 현대 도구 |
|---|---|---|---|
| Parker classical wind | §3 | Part 1: implicit isothermal solver | ENLIL, SWMF-IH |
| WKB wave-action conservation | §4 | Part 2: analytic amplitude growth | AWSoM, PENCIL-MHD |
| Wave-driven acceleration | §4 | Part 3: integrated momentum eq. | AWSoM, MAS+waves |
| Ion-cyclotron resonance | §6 | Part 4: gyrofrequency gap | Hybrid PIC codes |
| Elsässer variables | §7 | Part 5: synthetic Alfvénic fluc. | PSP/SolO analysis |
| Reflection-driven turbulence | §7 | Part 6: Q(r) profile | Cranmer-vanB 2005 code |
| Turbulence spectral scaling | §7 | Part 7: synthetic power-laws | Chen et al. PSP analysis |

**Key lessons / 핵심 교훈:**

1. **The Parker solution leaves a ~350 km/s gap for fast wind — wave pressure fills it naturally.** / Parker 해와 빠른 바람 사이 ~350 km/s 간극은 파동 압력으로 자연스럽게 메워진다.
2. **WKB amplitude growth comes for free from density drop.** / WKB 진폭 성장은 밀도 감소만으로도 발생.
3. **The Alfvén surface (where $u=v_A$) appears around 10–20 $R_\odot$ in our profiles** — PSP crossed it in 2021. / Alfvén 표면(${\sim}10$–$20R_\odot$)이 PSP가 2021년에 통과한 그 지점.
4. **Ion-cyclotron frequencies are 5–7 decades above injection** — a cascade is mandatory. / 이온-사이클로트론 주파수는 주입 대역보다 5–7 decade 위 — 캐스케이드 필수.
5. **Alfvénic correlation 1:1 between $\delta u$ and $\delta b$ is the single most distinctive in-situ signature.** / $\delta u$-$\delta b$ 1:1 상관은 in-situ Alfvén성의 가장 강력한 지표.
6. **Reflection makes $z^-$ out of $z^+$ — both must coexist for a cascade to heat.** / 반사가 $z^+$에서 $z^-$를 만들며, 두 성분의 **공존**이 가열의 필수 조건.
7. **Spectral slope alone doesn't discriminate Kol vs. IK** — higher-order statistics (structure functions, anisotropy) are needed. / 스펙트럼 기울기만으로는 K/IK 구분 불가 — 고차 통계 필요.

**Connections to next papers / 다음 논문과의 연결:**

- **LRSP #3 Marsch (2006)** — kinetic distribution functions sit at the end of this cascade.
- **LRSP #8 Cranmer (2009)** — companion review emphasizing the coronal heating side.
- **Cranmer & van Ballegooijen (2005) ApJS** — the reference implementation for reflection-driven turbulence.
- **Parker Solar Probe mission papers (Bale 2019+)** — first *in situ* verification.